# Audio Appendix: EDA and Model Evidence

This appendix expands the audio portion of the final prototype evidence notebook. The master reference remains `notebooks/00_final_prototype_evidence_notebook.ipynb`.

Purpose: document ASVspoof audio readiness, train/dev balance, MFCC/SVM metrics, behavior-model evidence, and runtime traceability without rerunning audio training.

In [ ]:
from pathlib import Path
import json
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

def project_path(relative: str) -> Path:
    return ROOT / relative

def load_json(relative: str) -> dict:
    path = project_path(relative)
    return json.loads(path.read_text(encoding="utf-8")) if path.exists() else {}

def status(path: Path) -> str:
    return "PASS" if path.exists() else "MISSING"

ROOT

## Dataset Readiness

Audio evidence uses ASVspoof-style bonafide/spoof data. Unlike email and transcript text, audio uses a train/dev validation style and signal features such as MFCC, spectral, energy, and timing statistics.

In [ ]:
labels_path = project_path("data/processed/audio/labels.csv")
audio_labels_df = pd.read_csv(labels_path) if labels_path.exists() else pd.DataFrame()

pd.DataFrame([
    {
        "Evidence item": "Processed audio labels",
        "Path": str(labels_path.relative_to(ROOT)),
        "Status": status(labels_path),
        "Rows": len(audio_labels_df),
        "Columns": len(audio_labels_df.columns),
        "Label column present": "label" in audio_labels_df.columns,
    }
])

In [ ]:
if not audio_labels_df.empty and "label" in audio_labels_df.columns:
    counts = audio_labels_df["label"].value_counts().sort_index()
    ax = counts.plot(kind="bar", figsize=(6, 4), color=["#0F766E", "#7C3AED"])
    ax.set_title("Audio Label Distribution")
    ax.set_xlabel("Label")
    ax.set_ylabel("Rows")
    plt.tight_layout()
else:
    print("Processed audio labels or label column is unavailable.")

## Audio Metric Evidence

The final audio evidence uses saved metric JSON files. The MFCC/statistical model provides the main audio authenticity signal, while the behavior Random Forest provides secondary behavior-feature evidence where available.

In [ ]:
audio_metrics = load_json("reports/metrics/audio_model_metrics.json")
behavior_metrics = load_json("reports/metrics/audio_behavior_metrics.json")

rows = []
for label, data in [("MFCC/statistical SVM", audio_metrics), ("Behavior Random Forest", behavior_metrics)]:
    metrics = data.get("metrics", {})
    rows.append({
        "Model": label,
        "Train samples": data.get("train_samples"),
        "Dev samples": data.get("dev_samples"),
        "Feature dimension": data.get("feature_dimension"),
        "Accuracy": metrics.get("accuracy"),
        "Precision": metrics.get("precision"),
        "Recall": metrics.get("recall"),
        "F1": metrics.get("f1"),
        "ROC-AUC": metrics.get("roc_auc"),
    })

pd.DataFrame(rows)

In [ ]:
importance_df = pd.DataFrame(behavior_metrics.get("feature_importances", []))
if not importance_df.empty:
    plot_df = importance_df.head(10).sort_values("importance", ascending=True)
    ax = plot_df.plot.barh(x="feature", y="importance", figsize=(8, 5), color="#2563EB", legend=False)
    ax.set_title("Top Audio Behavior Feature Importances")
    ax.set_xlabel("Importance")
    plt.tight_layout()
else:
    print("No saved behavior feature importances found.")

## Runtime Artifact And Source Traceability

The dashboard should load saved audio artifacts and combine audio concern with transcript/speech-quality signals where available. This notebook only verifies that the key evidence files exist.

In [ ]:
artifact_paths = [
    "models/audio_svm.pkl",
    "models/audio_behavior_rf.pkl",
    "reports/metrics/audio_model_metrics.json",
    "reports/metrics/audio_behavior_metrics.json",
    "app/transcript_tab.py",
    "src/audio/live_audio_analysis.py",
    "src/training/audio_trainer.py",
    "src/training/audio_behavior_trainer.py",
    "scripts/06_train_audio_model.py",
    "scripts/07_train_audio_behavior_model.py",
]

pd.DataFrame([
    {"Path": relative, "Status": status(project_path(relative))}
    for relative in artifact_paths
])

## Reviewer Note

Audio analysis can be affected by noise, compression, recording length, accents, overlapping speakers, and missing FFmpeg or Whisper dependencies. Treat audio output as an educational concern signal that supports human review.